# GATO — qualitative visualizations

Interactive, qualitative companion to the quantitative paper-figure scripts
(`reproduce_fig*.py`) in this folder. Where those scripts emit the published
plots/tables headlessly, this notebook *shows the trajectories*:

1. **Figure-8 EE tracking** (indy7) under a constant disturbance — how batch size
   sharpens disturbance rejection (qualitative companion to Fig-3 / `benchmark_fig8`).
2. **Pick-place** (iiwa14) with an unmodeled suspended payload — 3D end-effector
   trajectory + goal outcomes (qualitative companion to Fig-7 / `reproduce_fig7_pickplace`).

> **Requirements:** a CUDA GPU with the `bsqpN*` modules built, and the example extras
> (`pip install -e .[examples]` — pinocchio / matplotlib / meshcat). Run top-to-bottom
> from the **repo root**. Outputs are cleared in the committed copy; run to populate them.

_Consolidated from the legacy `gato_fig8_tracking.ipynb` + `gato_pickplace.ipynb`
(the only paper viz not covered by the headless `reproduce_*.py` scripts)._


## 1 · Figure-8 EE tracking under disturbance (indy7)


In [ ]:
import os, sys
import numpy as np
import pinocchio as pin
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3d projection)

# repo root, regardless of where the kernel started
REPO = os.path.abspath(os.path.join(os.getcwd()))
for _name in ("python", os.path.join("python", "bsqp")):
    p = os.path.join(REPO, _name)
    if p not in sys.path:
        sys.path.insert(0, p)

from bsqp.mpc_controller import MPC_GATO
from bsqp.common import figure8
from bsqp.config import FIG8_DEFAULT_PARAMS, INDY7_START_CONFIGS, BATCH_COLORS

np.random.seed(42)
print("imports complete")

In [ ]:
cfg = dict(
    batch_sizes=[1, 32, 128],
    N=32, dt=0.01,
    sim_time=16.0, sim_dt=0.001,
    start_config="zero",
    f_ext=np.array([0.0, 0.0, -60.0, 0.0, 0.0, 0.0]),  # constant -60 N along world Z
)
print(f"indy7 fig-8: batches={cfg['batch_sizes']}, N={cfg['N']}, dt={cfg['dt']}s, "
      f"{cfg['sim_time']}s @ {1/cfg['sim_dt']:.0f}Hz, f_ext_z={cfg['f_ext'][2]} N")

In [ ]:
urdf = os.path.join(REPO, "examples", "indy7_description", "indy7.urdf")
model_dir = os.path.dirname(urdf)
model, visual_model, collision_model = pin.buildModelsFromUrdf(urdf, model_dir)
print(f"indy7 loaded: {model.njoints-1} joints, nq={model.nq}, nv={model.nv}")

# reference figure-8 end-effector trajectory
fig8_traj = figure8(cfg["dt"], **FIG8_DEFAULT_PARAMS)
ref_points = fig8_traj.reshape(-1, 6)[:, :3]
print(f"reference: {len(ref_points)} points, {len(ref_points)*cfg['dt']:.1f}s")

In [ ]:
# Run the batched MPC once per batch size (GPU).
results = {}
x_start = np.hstack((INDY7_START_CONFIGS[cfg["start_config"]], np.zeros(6)))
for B in cfg["batch_sizes"]:
    mpc = MPC_GATO(model=model, model_path=urdf, N=cfg["N"], dt=cfg["dt"],
                   batch_size=B, constant_f_ext=cfg["f_ext"], track_full_stats=True)
    _, stats = mpc.run_mpc_fig8(x_start=x_start, fig8_traj=fig8_traj,
                                sim_dt=cfg["sim_dt"], sim_time=cfg["sim_time"])
    results[B] = stats
    print(f"  batch {B:>3}: {len(stats['timestamps'])} control steps")

In [ ]:
# Reference figure-8 (3D) + actual XZ tracking per batch size.
fig = plt.figure(figsize=(13, 4))

ax0 = fig.add_subplot(1, len(results)+1, 1, projection="3d")
ax0.plot(ref_points[:, 0], ref_points[:, 1], ref_points[:, 2], "r-", alpha=0.6)
ax0.set_title("reference (3D)"); ax0.set_xlabel("X"); ax0.set_ylabel("Y"); ax0.set_zlabel("Z")
ax0.view_init(elev=20, azim=135)

for j, B in enumerate(sorted(results), start=2):
    ax = fig.add_subplot(1, len(results)+1, j)
    ax.plot(ref_points[:, 0], ref_points[:, 2], ":", lw=1.0, alpha=0.5, label="reference")
    ee = results[B]["ee_actual"]
    ax.plot(ee[:, 0], ee[:, 2], color=BATCH_COLORS.get(B, "#000"), lw=1.5,
            alpha=0.85, label=f"batch {B}")
    ax.set_title(f"batch {B}"); ax.set_xlabel("X [m]")
    if j == 2: ax.set_ylabel("Z [m]")
    ax.set_aspect("equal", "box"); ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.7, 0.0); ax.set_ylim(0.5, 1.1); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()
print("Larger batches track the figure-8 more tightly under the constant disturbance.")

## 2 · Pick-place with an unmodeled suspended payload (iiwa14)

A 15 kg pendulum hangs off the end-effector (unmodeled in the planner). GATO samples
candidate disturbances across the batch and tracks the best — larger batches reach more
goals. Below: the 3D EE trajectory + goal markers, and a success/time/solve-time table.


In [ ]:
from bsqp.config import (PICKPLACE_DEFAULT_GOALS, PENDULUM_DEFAULT_PARAMS,
                         IIWA14_START_CONFIGS)

urdf_i = os.path.join(REPO, "examples", "iiwa_description", "iiwa14.urdf")
model_i, vmodel_i, cmodel_i = pin.buildModelsFromUrdf(urdf_i, os.path.dirname(urdf_i))

pp = dict(N=16, dt=0.01, sim_dt=0.001,
          batch_sizes=[1, 8, 32, 128],
          goals=PICKPLACE_DEFAULT_GOALS,
          pendulum=PENDULUM_DEFAULT_PARAMS.copy())
x_start_i = np.hstack((IIWA14_START_CONFIGS["zero"], np.zeros(7)))
print(f"iiwa14 pick-place: {len(pp['goals'])} goals, N={pp['N']}, "
      f"pendulum {pp['pendulum']['mass']}kg / {pp['pendulum']['length']}m")

In [ ]:
# Run pick-place per batch size (GPU). batch=1 has no force estimation.
pp_results = {}
for B in pp["batch_sizes"]:
    mpc = MPC_GATO(model=model_i, model_path=urdf_i, N=pp["N"], dt=pp["dt"],
                   batch_size=B, pendulum_config=pp["pendulum"], track_full_stats=True)
    _, stats = mpc.run_mpc_goals(x_start=x_start_i, goals=pp["goals"], sim_dt=pp["sim_dt"])
    pp_results[B] = stats
    reached = sum(o == "reached" for o in stats["goal_outcomes"])
    print(f"  batch {B:>3}: {reached}/{len(pp['goals'])} goals reached")

In [ ]:
# 3D end-effector trajectory (highest batch) + goal markers colored by outcome.
B_show = max(pp_results)
stats = pp_results[B_show]
ee = np.asarray(stats["ee_actual"])
goals = np.asarray(pp["goals"])

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot(ee[:, 0], ee[:, 1], ee[:, 2], "-", lw=1.2, alpha=0.8, color=BATCH_COLORS.get(B_show, "#000"),
        label=f"EE path (batch {B_show})")
for i, (gpos, outcome) in enumerate(zip(goals, stats["goal_outcomes"])):
    c = "tab:green" if outcome == "reached" else "tab:red"
    ax.scatter(*gpos, c=c, s=70, marker="o", edgecolors="k", depthshade=False,
               label=("reached" if c == "tab:green" else "missed") if i < 2 else None)
ax.set_xlabel("X [m]"); ax.set_ylabel("Y [m]"); ax.set_zlabel("Z [m]")
ax.set_title(f"iiwa14 pick-place EE trajectory (batch {B_show})")
ax.view_init(elev=20, azim=135); ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Summary: goals reached / time-to-complete / mean solve time per batch size.
ngoals = len(pp["goals"])
print(f"{'batch':<8}{'goals':<10}{'time (s)':<12}{'mean solve (ms)':<18}")
print("-" * 48)
for B in sorted(pp_results):
    s = pp_results[B]
    reached = sum(o == "reached" for o in s["goal_outcomes"])
    t = s.get("time_to_all_reached")
    tstr = f"{t:.2f}" if t is not None else "timeout"
    solve = np.asarray(s["solve_times"]) * 1e3
    goals_str = f"{reached}/{ngoals}"
    solve_str = f"{solve.mean():.3f} ± {solve.std():.3f}"
    print(f"{B:<8}{goals_str:<10}{tstr:<12}{solve_str:<18}")